<a href="https://colab.research.google.com/github/ayushniet01/NLP-Program-using-Python-NLTK-and-spaCy/blob/main/Information%20Retrieval%20System%20with%20Ranking%20using%20TF-IDF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Information Retrieval System with Ranking using TF-IDF



In [1]:
import math
import numpy as np
import pandas as pd
import re
from collections import Counter

## 1. Define Sample Documents
Let's define a small corpus of documents to index and search.

In [2]:
documents = {
    "doc1": "The quick brown fox jumps over the lazy dog.",
    "doc2": "Never jump over the lazy dog quickly.",
    "doc3": "The brown fox is quick and clever.",
    "doc4": "Lazy dogs love to sleep all day in the sun."
}

for doc_id, text in documents.items():
    print(f"{doc_id}: {text}")

doc1: The quick brown fox jumps over the lazy dog.
doc2: Never jump over the lazy dog quickly.
doc3: The brown fox is quick and clever.
doc4: Lazy dogs love to sleep all day in the sun.


## 2. Preprocessing
We need to tokenize, lowercase, and clean the text.

In [3]:
def preprocess(text):
    # Lowercase and keep only alphanumeric characters
    tokens = re.findall(r'\b\w+\b', text.lower())
    return tokens

processed_docs = {doc_id: preprocess(text) for doc_id, text in documents.items()}
print("Processed Documents:")
for doc_id, tokens in processed_docs.items():
    print(f"{doc_id}: {tokens}")

Processed Documents:
doc1: ['the', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog']
doc2: ['never', 'jump', 'over', 'the', 'lazy', 'dog', 'quickly']
doc3: ['the', 'brown', 'fox', 'is', 'quick', 'and', 'clever']
doc4: ['lazy', 'dogs', 'love', 'to', 'sleep', 'all', 'day', 'in', 'the', 'sun']


## 3. Compute TF-IDF
We will construct the vocabulary and compute term frequency (TF) and inverse document frequency (IDF).

In [4]:
# Build Vocabulary
vocabulary = sorted(list(set(token for tokens in processed_docs.values() for token in tokens)))
print(f"Vocabulary size: {len(vocabulary)}")
print("Vocabulary:", vocabulary)

# Calculate IDF
num_docs = len(documents)
idf = {}
for term in vocabulary:
    # Count how many documents contain this term
    doc_count = sum(1 for tokens in processed_docs.values() if term in tokens)
    # Apply smooth IDF formula
    idf[term] = math.log((1 + num_docs) / (1 + doc_count)) + 1

# Calculate TF-IDF Vectors for Documents
def get_tfidf_vector(tokens):
    counts = Counter(tokens)
    total_terms = len(tokens)
    vector = []
    for term in vocabulary:
        # Term Frequency (TF)
        tf = counts.get(term, 0) / total_terms if total_terms > 0 else 0
        # TF-IDF
        vector.append(tf * idf[term])
    return np.array(vector)

doc_vectors = {doc_id: get_tfidf_vector(tokens) for doc_id, tokens in processed_docs.items()}

# Create a DataFrame to visualize Document Vectors
df_vectors = pd.DataFrame(doc_vectors, index=vocabulary)
display(df_vectors)

Vocabulary size: 22
Vocabulary: ['all', 'and', 'brown', 'clever', 'day', 'dog', 'dogs', 'fox', 'in', 'is', 'jump', 'jumps', 'lazy', 'love', 'never', 'over', 'quick', 'quickly', 'sleep', 'sun', 'the', 'to']


,doc1,doc2,doc3,doc4
all,0.000000,0.000000,0.000000,0.191629
and,0.000000,0.000000,0.273756,0.000000
brown,0.167870,0.000000,0.215832,0.000000
clever,0.000000,0.000000,0.273756,0.000000
day,0.000000,0.000000,0.000000,0.191629
dog,0.167870,0.215832,0.000000,0.000000
dogs,0.000000,0.000000,0.000000,0.191629
fox,0.167870,0.000000,0.215832,0.000000
in,0.000000,0.000000,0.000000,0.191629
is,0.000000,0.000000,0.273756,0.000000


## 4. Querying and Ranking
To find relevant documents, we preprocess the query, construct its TF-IDF vector, and measure the cosine similarity between the query vector and all document vectors.

In [5]:
def cosine_similarity(v1, v2):
    dot_product = np.dot(v1, v2)
    norm_v1 = np.linalg.norm(v1)
    norm_v2 = np.linalg.norm(v2)
    if norm_v1 == 0 or norm_v2 == 0:
        return 0.0
    return dot_product / (norm_v1 * norm_v2)

def search(query, top_n=5):
    query_tokens = preprocess(query)
    query_vector = get_tfidf_vector(query_tokens)

    scores = {}
    for doc_id, doc_vector in doc_vectors.items():
        scores[doc_id] = cosine_similarity(query_vector, doc_vector)

    # Sort by score in descending order
    ranked_results = sorted(scores.items(), key=lambda x: x[1], reverse=True)

    print(f"Query: '{query}'\n")
    print("Rank  Document ID  Score    Content")
    print("-" * 70)
    for rank, (doc_id, score) in enumerate(ranked_results[:top_n], 1):
        if score > 0:
            print(f"{rank:<5} {doc_id:<12} {score:.4f}   {documents[doc_id]}")
        else:
            print(f"{rank:<5} {doc_id:<12} {score:.4f}   [No match]")

# Test Search
search("quick brown fox")

Query: 'quick brown fox'

Rank  Document ID  Score    Content
----------------------------------------------------------------------
1     doc3         0.6025   The brown fox is quick and clever.
2     doc1         0.5768   The quick brown fox jumps over the lazy dog.
3     doc2         0.0000   [No match]
4     doc4         0.0000   [No match]
